# Phase 7 — FAISS + RAG Evidence Retrieval

This notebook walks through the retrieval pipeline step by step: load reference documents → chunk → embed → build a FAISS index → retrieve passages for claims.

**Read this first**

* The reference documents used here are **synthetic demonstration documents** (fictional towns, numbers and events) shipped in `data/rag/sample_reference_docs/`. They are **not real-world evidence**.
* A retrieval **score is a cosine similarity** between text embeddings. It is **not** a probability and does **not** show that a claim is true or false. A passage can be highly similar to a claim and still contradict it.
* Phase 7 never labels a claim as verified or false. That is outside this phase.
* Outputs are intentionally cleared in the committed notebook; run the cells yourself.

In [ ]:
import sys, json, tempfile
from pathlib import Path

# Make the project root importable whether the notebook is run from notebooks/ or the root
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.rag.config import SAMPLE_DOCS_DIR
from src.rag.document_loader import load_documents
from src.rag.text_chunker import chunk_documents
from src.rag.embeddings import SentenceTransformerEmbedder, HashingEmbedder, EmbeddingError
from src.rag.faiss_index import EvidenceIndex, build_or_update_index
from src.rag.retriever import EvidenceRetriever
print("Sample documents folder:", SAMPLE_DOCS_DIR)

## 1. Load the (synthetic) reference documents

In [ ]:
report = load_documents(SAMPLE_DOCS_DIR)
for doc in report.documents:
    print(f"{doc.source_id:45s} pages={len(doc.pages)}  chars={sum(len(p.text) for p in doc.pages)}  sha256={doc.content_hash[:12]}...")
print("Skipped:", [(s.source_id, s.reason) for s in report.skipped])

## 2. Split into overlapping chunks
Chunk size and overlap are in characters and configurable. Every chunk keeps its source filename, page (None for `.txt`) and chunk id.

In [ ]:
chunks = chunk_documents(report.documents, chunk_size=300, overlap=60)
print(len(chunks), "chunks")
pd.DataFrame([{"chunk_id": c.chunk_id, "file": c.filename, "page": c.page, "chars": len(c.text), "text": c.text[:70] + "..."} for c in chunks]).head(8)

## 3. Choose the embedding model

The real model is `sentence-transformers/all-MiniLM-L6-v2` (about 90 MB, downloaded once on first use, then cached; no API key). 

If the model cannot be loaded here (for example no internet on the first run), the cell below falls back to a **lexical `HashingEmbedder` that matches words only and has no semantic understanding**. It exists so the notebook still runs offline. Retrieval quality with it is **not representative** of the real model.

In [ ]:
try:
    embedder = SentenceTransformerEmbedder()
    embedder.embed_query("test")          # forces the model to load now
    USING_REAL_MODEL = True
except EmbeddingError as err:
    print("Could not load the sentence-transformers model:", err)
    embedder = HashingEmbedder()
    USING_REAL_MODEL = False
print("Embedder:", embedder.model_name, "| real semantic model:", USING_REAL_MODEL)
if not USING_REAL_MODEL:
    print("WARNING: using the word-overlap demo embedder; do not treat these results as semantic retrieval.")

vectors = embedder.embed_texts([c.text for c in chunks])
print("Embedding matrix shape:", vectors.shape, "| dtype:", vectors.dtype)
print("Vector norms (should all be ~1.0):", np.round(np.linalg.norm(vectors, axis=1)[:5], 4))

## 4. Build, save and reload the FAISS index
The index is written to a temporary folder so this demo does not touch your real index.

In [ ]:
demo_dir = Path(tempfile.mkdtemp(prefix="rag_demo_"))
result = build_or_update_index(SAMPLE_DOCS_DIR, demo_dir / "index", embedder, chunk_size=300, chunk_overlap=60)
print("Status:", result.status, "| chunks:", result.index.size, "| dimension:", result.index.dimension)
print("Files:", sorted(p.name for p in (demo_dir / "index").iterdir()))

# Running it again with unchanged documents re-uses the saved index instead of re-embedding
again = build_or_update_index(SAMPLE_DOCS_DIR, demo_dir / "index", embedder, chunk_size=300, chunk_overlap=60)
print("Second run status:", again.status)

reloaded = EvidenceIndex.load(demo_dir / "index", expected_model_name=embedder.model_name)
print("Reloaded vectors:", reloaded.index.ntotal, "| metadata records:", len(reloaded.chunks))

## 5. Retrieve evidence for sample claims
Each claim is searched independently. `min_score` is a heuristic similarity cut-off, **not** a calibrated value; tune it for your own reference collection.

In [ ]:
retriever = EvidenceRetriever(reloaded, embedder, top_k=3, min_score=0.35 if USING_REAL_MODEL else 0.2)

claims = [
    "The Alder River levee was completed in 1990.",
    "Library membership is free for all residents.",
    "A single tram ticket costs two dollars.",
]
results = retriever.retrieve_many(claims)

rows = []
for res in results:
    for ev in res["retrieved_evidence"]:
        rows.append({"claim": res["claim"], "rank": ev["rank"], "score (cosine sim.)": ev["score"],
                     "source": ev["source"], "page": ev["page"], "chunk_id": ev["chunk_id"], "passage": ev["text"][:90] + "..."})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("Statuses:", [r["retrieval_status"] for r in results])

## 6. Source metadata and the limitation notes
The source filename, page and chunk id come straight from stored metadata; nothing is invented (the sample `.txt` files have no page numbers, so `page` is `None`).

In [ ]:
print(json.dumps(results[0], indent=2)[:1800])

## 7. A claim the reference collection does not cover
When no passage reaches the threshold, the status is `no_relevant_evidence` and **no evidence is made up**. This does not mean the claim is true or false.

In [ ]:
unrelated = retriever.retrieve("Scientists discovered a new species of deep-sea octopus.")
print("Status:", unrelated["retrieval_status"])
print("Evidence:", unrelated["retrieved_evidence"])
print("Limitations:")
for line in unrelated["limitations"]:
    print(" -", line)

# Other statuses: an empty claim, and a missing index
print(retriever.retrieve("   ")["retrieval_status"])
print(EvidenceRetriever.from_directory(demo_dir / "missing", embedder).retrieve("any claim")["retrieval_status"])

## 8. Claims → evidence

`gather_evidence_for_article` takes an article, uses the claims you supply (or, if none are given, picks candidate sentences with a simple rule-based selector), and retrieves evidence for each claim. It keeps the ML prediction and the retrieved evidence **separate**.

In this demo we pass **manually supplied claims**.

In [ ]:
from src.rag.evidence_pipeline import gather_evidence_for_article

article = "The council said the Alder River levee was finished in 1990. Officials also said library membership is free."
output = gather_evidence_for_article(
    article, retriever,
    manual_claims=["The Alder River levee was completed in 1990.", "Library membership is free for all residents."],
    article_metadata={"title": "Demo article (synthetic)"},
)
print("Keys:", list(output))
print("Claim source:", output["claim_source"], "| ml_prediction:", output["ml_prediction"])
for c in output["claims"]:
    print("-", c["claim"], "->", c["retrieval_status"], [e["source"] for e in c["retrieved_evidence"]])


## Summary and limitations

* Retrieval returns the most *similar* passages, not proof. The score is a cosine similarity, not a truth probability.
* Results are only as good as the supplied reference documents, which can be incomplete, outdated or wrong.
* A similar passage may support, contradict or merely mention a claim. Judging which is a later step and is **not** done in Phase 7.